# US Fiscal Research Database

4 CSV outputs · Quarterly fiscal–macro, monthly debt, CBO expectations, CBO news

In [ ]:
"""Build reusable US fiscal, macro, SPF and CBO vintage panels.
Run from C:\\Users\\gayeon\\Dropbox\\key_data\\Fiscal.
Reads only local sources; never substitutes simulated or forward-filled data.
"""
from pathlib import Path
import os
import warnings
import numpy as np
import pandas as pd

ROOT = Path(os.environ.get('FISCAL_ROOT', r'C:\Users\gayeon\Dropbox\key_data\Fiscal'))
RAW, OUT = ROOT / 'raw', ROOT / 'output' / 'core'
OUT.mkdir(parents=True, exist_ok=True)
LOG, PANELS, DEFINITIONS = [], {}, []


def source(name):
    hits = [p for p in RAW.glob('*') if p.is_file() and p.stem.casefold() == name.casefold()] if RAW.exists() else []
    if not hits and name.startswith('cbo_'):
        hits = [p for p in ROOT.glob('*') if p.is_file() and p.stem.casefold() == name.casefold()]
    priority = {'.csv': 0, '.xlsx': 1, '.xls': 2}
    return sorted(hits, key=lambda p: priority.get(p.suffix.lower(), 9))[0] if hits else None


def read(name, sheet=0, header=0):
    p = source(name)
    if p is None:
        LOG.append({'source': name, 'path': '', 'status': 'MISSING', 'rows': 0})
        return None
    if p.suffix.lower() == '.csv':
        df = pd.read_csv(p, encoding='utf-8-sig', low_memory=False)
    else:
        df = pd.read_excel(p, sheet_name=sheet, header=header)
    df.columns = [str(c).strip() for c in df.columns]
    LOG.append({'source': name, 'path': str(p), 'status': 'LOADED', 'rows': len(df)})
    return df


def quarterize(df, date_col='observation_date'):
    if df is None: return None
    out = df.copy()
    if date_col not in out.columns:
        date_col = next((c for c in ['date', 'DATE', 'observation_date'] if c in out), None)
    if date_col is None: raise KeyError('No observation date column: ' + repr(out.columns.tolist()))
    out['date'] = pd.to_datetime(out[date_col], errors='coerce')
    out = out.dropna(subset=['date'])
    out['quarter'] = out['date'].dt.to_period('Q')
    return out.sort_values('date')


def series(name, code, target, how='last'):
    df = quarterize(read(name))
    if df is None: return None
    if code not in df.columns:
        raise KeyError(f'{name} missing expected column {code}; columns={df.columns.tolist()[:20]}')
    df[target] = pd.to_numeric(df[code].replace('.', np.nan), errors='coerce')
    agg = 'mean' if how == 'mean' else 'last'
    return df.groupby('quarter', as_index=False)[target].agg(agg)


def merge_many(frames):
    frames = [f for f in frames if f is not None]
    if not frames: return None
    out = frames[0].copy()
    for frame in frames[1:]:
        out = out.merge(frame, on='quarter', how='outer', validate='one_to_one')
    return out.sort_values('quarter').reset_index(drop=True)


def safe_div(a, b, multiplier=1):
    b = pd.to_numeric(b, errors='coerce')
    return multiplier * pd.to_numeric(a, errors='coerce') / b.where(b != 0)


def save(name, df, key=None):
    if df is None or df.empty:
        return
    if key and df.duplicated(key).any():
        raise ValueError(f'{name}: duplicate key {key}')
    PANELS[name] = df.copy()


def note(panel, variable, definition, unit='', source_name='', caution=''):
    DEFINITIONS.append(dict(panel=panel, variable=variable, definition=definition, unit=unit,
                            source=source_name, caution=caution))


# 01. MACRO: CBO output gap, inflation, interest rates
ngdp = series('bea_nominal_gdp', 'GDP', 'nominal_gdp_bn_saar')
rgdp = series('bea_real_gdp', 'GDPC1', 'real_gdp_bn_saar')
potential = series('cbo_real_potential_gdp', 'GDPPOT', 'potential_real_gdp_bn_saar')
cpi = series('bea_cpi', 'CPIAUCSL', 'cpi_index', 'mean')
dff = series('fed_policy_rates', 'DFF', 'effective_ffr_pct', 'mean')
macro = merge_many([ngdp, rgdp, potential, cpi, dff])
if macro is not None:
    if {'real_gdp_bn_saar', 'potential_real_gdp_bn_saar'} <= set(macro):
        macro['output_gap_cbo_pct'] = 100*(safe_div(macro.real_gdp_bn_saar, macro.potential_real_gdp_bn_saar)-1)
    if 'cpi_index' in macro:
        macro['cpi_inflation_qoq_ann_pct'] = 100*((macro.cpi_index/macro.cpi_index.shift(1))**4-1)
        macro['cpi_inflation_yoy_pct'] = 100*(macro.cpi_index/macro.cpi_index.shift(4)-1)
    if 'real_gdp_bn_saar' in macro:
        macro['real_gdp_growth_qoq_ann_pct'] = 100*((macro.real_gdp_bn_saar/macro.real_gdp_bn_saar.shift(1))**4-1)
    for stem in ['tb3ms', 'fred_tb3ms', 'tbill_3m']:
        if source(stem):
            tb = series(stem, 'TB3MS', 'tbill_3m_pct', 'mean')
            macro = macro.merge(tb, on='quarter', how='left', validate='one_to_one')
            break
    macro['period'] = macro.quarter.astype(str)
    save('01_macro_quarterly', macro, ['quarter'])
    note('01_macro_quarterly','output_gap_cbo_pct','100*(Real GDP / CBO potential GDP - 1)','percent','BEA GDPC1 + CBO GDPPOT')
    note('01_macro_quarterly','cpi_inflation_qoq_ann_pct','100*((quarterly mean CPI / lag CPI)^4 - 1)','percent annualized','CPIAUCSL','Ex-post inflation, not survey expectation')


# 02. FISCAL: BEA national-account definition, not Treasury cash deficit
net = series('bea_federal_net_lending', 'AD02RC1Q027SBEA', 'federal_net_lending_bn_saar')
interest = series('bea_federal_interest_payments', 'A091RC1Q027SBEA', 'federal_interest_payments_bn_saar')
flows = merge_many([ngdp, net, interest])
if flows is not None and {'federal_net_lending_bn_saar', 'federal_interest_payments_bn_saar'} <= set(flows):
    flows['primary_surplus_bn_saar'] = flows.federal_net_lending_bn_saar + flows.federal_interest_payments_bn_saar
    flows['primary_deficit_bn_saar'] = -flows.primary_surplus_bn_saar
    flows['primary_surplus_bn_quarter'] = flows.primary_surplus_bn_saar/4
    flows['primary_deficit_bn_quarter'] = flows.primary_deficit_bn_saar/4
    if 'nominal_gdp_bn_saar' in flows:
        flows['primary_surplus_pct_gdp'] = safe_div(flows.primary_surplus_bn_saar, flows.nominal_gdp_bn_saar, 100)
        flows['primary_deficit_pct_gdp'] = -flows.primary_surplus_pct_gdp
        flows['interest_payments_pct_gdp'] = safe_div(flows.federal_interest_payments_bn_saar, flows.nominal_gdp_bn_saar, 100)
    flows['period'] = flows.quarter.astype(str)
    save('02_fiscal_flows_quarterly', flows, ['quarter'])
    note('02_fiscal_flows_quarterly','primary_surplus_bn_saar','Federal net lending + federal interest payments','USD billion, SAAR','BEA','NIPA accrual concept, not unified-budget cash balance')
    note('02_fiscal_flows_quarterly','primary_surplus_bn_quarter','Primary surplus SAAR / 4','USD billion per quarter','BEA')


# 03. DALLAS MONTHLY DEBT: preserve month AND end-of-quarter stock
raw_debt = source('dallas_govdebt')
debt_m = None
if raw_debt is not None:
    if raw_debt.suffix.lower() in ('.xls', '.xlsx'):
        debt_m = pd.read_excel(raw_debt, sheet_name='data', header=[1,2])
        if debt_m.shape[1] != 7:
            raise ValueError('Dallas debt workbook schema changed; expected 7 columns')
        debt_m.columns = ['date','par_gross_debt_bn','par_private_debt_bn','par_marketable_debt_bn',
                          'mv_gross_debt_bn','mv_private_debt_bn','mv_marketable_debt_bn']
    else:
        debt_m = pd.read_csv(raw_debt)
        if len(debt_m.columns)==7 and 'date' not in debt_m:
            debt_m.columns=['date','par_gross_debt_bn','par_private_debt_bn','par_marketable_debt_bn',
                            'mv_gross_debt_bn','mv_private_debt_bn','mv_marketable_debt_bn']
    debt_m=quarterize(debt_m, 'date')
    for col in debt_m.columns:
        if col.endswith('_bn'): debt_m[col]=pd.to_numeric(debt_m[col], errors='coerce')
    debt_m['mv_private_to_par_private'] = safe_div(debt_m.mv_private_debt_bn,debt_m.par_private_debt_bn)
    debt_m['mv_marketable_to_par_marketable'] = safe_div(debt_m.mv_marketable_debt_bn,debt_m.par_marketable_debt_bn)
    save('03_debt_monthly', debt_m, ['date'])
    debt_q = debt_m.sort_values('date').groupby('quarter', as_index=False).tail(1).copy()
    debt_q=debt_q.rename(columns={'date':'debt_observation_month'}).sort_values('quarter').reset_index(drop=True)
    if ngdp is not None: debt_q=debt_q.merge(ngdp,on='quarter',how='left',validate='one_to_one')
    if 'nominal_gdp_bn_saar' in debt_q:
        for stem in ('gross','private','marketable'):
            debt_q[f'mv_{stem}_debt_pct_gdp']=safe_div(debt_q[f'mv_{stem}_debt_bn'],debt_q.nominal_gdp_bn_saar,100)
    if flows is not None and 'primary_surplus_bn_quarter' in flows:
        debt_q=debt_q.merge(flows[['quarter','primary_surplus_bn_quarter','primary_surplus_bn_saar']],on='quarter',how='left',validate='one_to_one')
        debt_q['primary_surplus_pct_mv_private_debt_q']=safe_div(debt_q.primary_surplus_bn_quarter, debt_q.mv_private_debt_bn,100)
        debt_q['primary_surplus_pct_mv_private_debt_ann']=safe_div(debt_q.primary_surplus_bn_saar, debt_q.mv_private_debt_bn,100)
    debt_q['period']=debt_q.quarter.astype(str)
    save('04_debt_quarterly', debt_q,['quarter'])
    note('04_debt_quarterly','mv_private_debt_pct_gdp','100*End-quarter MV privately held federal debt / nominal GDP SAAR','percent','Dallas Fed + BEA')
    note('04_debt_quarterly','primary_surplus_pct_mv_private_debt_q','100*(Primary surplus SAAR/4)/End-quarter MV privately held debt','percent per quarter','BEA + Dallas Fed')
    note('03_debt_monthly','mv_private_to_par_private','Market value / par value of privately held federal debt','ratio','Dallas Fed','Bond valuation ratio, not duration or term premium')


# 04. SPF: horizon-preserving long panels from sheets + separate inflation summary

def spf_sheet(df):
    lookup={c.upper():c for c in df.columns}
    if not {'YEAR','QUARTER'} <= set(lookup): return None
    df=df.copy()
    df['YEAR']=pd.to_numeric(df[lookup['YEAR']],errors='coerce')
    df['QUARTER']=pd.to_numeric(df[lookup['QUARTER']],errors='coerce')
    df=df.loc[df.YEAR.between(1900,2100)&df.QUARTER.between(1,4)].copy()
    df['survey_quarter']=pd.PeriodIndex(df.YEAR.astype(int).astype(str)+'Q'+df.QUARTER.astype(int).astype(str),freq='Q')
    return df


def read_spf_workbook(stem, individual=False):
    path=source(stem)
    if path is None or path.suffix.lower() not in ('.xls','.xlsx'): return None
    xls=pd.ExcelFile(path)
    allowed=['NGDP','RGDP','CPI','CORECPI','PCE','TBILL','TBOND','CPI10','PCE10']
    frames=[]
    for variable in allowed:
        if variable not in xls.sheet_names: continue
        df=spf_sheet(pd.read_excel(path,sheet_name=variable))
        if df is None: continue
        index=['survey_quarter','YEAR','QUARTER']
        if individual: index += [c for c in ['ID','INDUSTRY'] if c in df]
        valcols=[c for c in df if c not in index and c.upper().startswith(variable)]
        if not valcols: continue
        z=df.melt(id_vars=index,value_vars=valcols,var_name='raw_horizon',value_name='forecast_value')
        z['variable']=variable
        z['horizon']=z.raw_horizon.str[len(variable):]
        z['forecast_value']=pd.to_numeric(z.forecast_value,errors='coerce')
        frames.append(z.dropna(subset=['forecast_value']))
    if not frames: return None
    out=pd.concat(frames,ignore_index=True)
    out['survey_quarter']=out.survey_quarter.astype(str)
    return out.sort_values(['survey_quarter','variable','raw_horizon']).reset_index(drop=True)

spf_mean=read_spf_workbook('spf_mean_level')
spf_ind=read_spf_workbook('spf_microdata',individual=True)
save('05_spf_mean_horizons',spf_mean)
save('06_spf_individual_horizons',spf_ind)
spf_summary=read('spf_inflation_short_long', sheet='INFLATION')
if spf_summary is not None:
    spf_summary=spf_sheet(spf_summary)
    if spf_summary is not None:
        for code, target in [('INFCPI1YR','spf_cpi_inflation_1yr_pct'),('INFCPI10YR','spf_cpi_inflation_10yr_pct'),
                             ('INFPGDP1YR','spf_gdp_deflator_inflation_1yr_pct')]:
            if code in spf_summary:
                spf_summary[target]=pd.to_numeric(spf_summary[code],errors='coerce')
        spf_summary['period']=spf_summary.survey_quarter.astype(str)
        save('07_spf_inflation_quarterly',spf_summary,['survey_quarter'])
        note('07_spf_inflation_quarterly','spf_cpi_inflation_10yr_pct','Long-run CPI inflation survey mean (as named in SPF source)','percent annualized','Philadelphia Fed SPF','Not a 10Y Treasury bill forecast')


# 05. CBO: forecasts retain vintage x fiscal year x component and forecast errors
expect=read('cbo_expectations')
if expect is None:
    base, actuals, actualgdp=read('cbo_baselines'),read('cbo_actuals'),read('cbo_actual_GDP')
    if base is not None:
        expect=base.rename(columns={'value':'baseline_value_bn','projected_fiscal_year':'fiscal_year'})
        keys=['component','category','subcategory','fiscal_year']
        if actuals is not None: expect=expect.merge(actuals,on=keys,how='left')
        if actualgdp is not None:
            if 'GDP' in actualgdp: actualgdp=actualgdp.rename(columns={'GDP':'actual_gdp_bn'})
            expect=expect.merge(actualgdp,on='fiscal_year',how='left')
        if {'actual_value','baseline_value_bn'} <= set(expect):
            expect['forecast_error_bn']=expect.actual_value-expect.baseline_value_bn
if expect is not None:
    expect['baseline_date']=pd.to_datetime(expect.baseline_date,errors='coerce')
    expect=expect.dropna(subset=['baseline_date']).copy()
    expect['fiscal_year']=pd.to_numeric(expect.fiscal_year, errors='coerce')
    expect['horizon_fy']=expect.fiscal_year-expect.baseline_date.dt.year
    expect['vintage_quarter']=expect.baseline_date.dt.to_period('Q').astype(str)
    if {'actual_value','baseline_value_bn'} <= set(expect):
        expect['forecast_error_bn']=expect.actual_value-expect.baseline_value_bn
    save('08_cbo_vintage_forecasts',expect,['component','category','subcategory','baseline_date','fiscal_year'])
    note('08_cbo_vintage_forecasts','baseline_value_bn','CBO forecast by publication vintage, fiscal year and component','USD bn','CBO','Never merge future realized actuals into ex-ante forecasts for causal identification')
    note('08_cbo_vintage_forecasts','baseline_value_pct_gdp','CBO predicted fiscal amount / ex-post actual GDP','percent','CBO','Ex-post denominator: not ex-ante forecast-to-forecast ratio')
    total=expect.loc[(expect.category=='Total')&(expect.subcategory=='Total')].copy()
    if not total.empty:
        save('09_cbo_total_vintage_forecasts',total,['component','baseline_date','fiscal_year'])
        deficit=total[total.component=='deficit'].copy()
        save('10_cbo_deficit_vintages',deficit,['baseline_date','fiscal_year'])


# 06. CBO NEWS: keep the fiscal-year-by-vintage panel, with sign conventions
news=read('cbo_news')
if news is None:
    changes=read('cbo_baseline_changes')
    if changes is not None:
        idx=['component','category','subcategory','changes_baseline_date','projected_fiscal_year']
        news=changes.pivot_table(index=idx,columns='change_category',values='value',aggfunc='sum',fill_value=0).reset_index()
        news.columns.name=None
        news=news.rename(columns={'Legislative':'change_legislative_bn','Economic':'change_economic_bn',
                                  'Technical':'change_technical_bn'})
if news is not None:
    news['changes_baseline_date']=pd.to_datetime(news.changes_baseline_date,errors='coerce')
    news=news.dropna(subset=['changes_baseline_date']).copy()
    for c in ['change_legislative_bn','change_economic_bn','change_technical_bn']:
        if c not in news: news[c]=0.0
    news['policy_news_bn']=news.change_legislative_bn
    news['nonpolicy_news_bn']=news.change_economic_bn+news.change_technical_bn
    news['total_revision_bn']=news.policy_news_bn+news.nonpolicy_news_bn
    news['horizon_fy']=pd.to_numeric(news.projected_fiscal_year,errors='coerce')-news.changes_baseline_date.dt.year
    news['announcement_quarter']=news.changes_baseline_date.dt.to_period('Q').astype(str)
    assert np.allclose(news.total_revision_bn,news[['change_legislative_bn','change_economic_bn','change_technical_bn']].sum(axis=1))
    save('11_cbo_news_vintage_fy',news,['component','category','subcategory','changes_baseline_date','projected_fiscal_year'])
    n_total=news.loc[(news.category=='Total')&(news.subcategory=='Total')].copy()
    save('12_cbo_total_news_vintage_fy',n_total,['component','changes_baseline_date','projected_fiscal_year'])
    note('11_cbo_news_vintage_fy','change_legislative_bn','Legislative revision to a fiscal component for target FY','USD bn','CBO')
    note('11_cbo_news_vintage_fy','change_economic_bn','Economic revision, distinct from legislative news','USD bn','CBO')
    note('11_cbo_news_vintage_fy','change_technical_bn','Technical revision','USD bn','CBO')

    # 07. FISCAL TIMING: aggregate deficit news by publication vintage/horizon without look-ahead.
    deficit_news=n_total.loc[n_total.component=='deficit'].copy()
    if not deficit_news.empty:
        deficit_news=deficit_news.loc[deficit_news.horizon_fy.ge(0)].copy()
        deficit_news['horizon_group']=pd.cut(deficit_news.horizon_fy,bins=[-0.5,1.5,5.5,10.5,np.inf],
            labels=['h0_1','h2_5','h6_10','h11plus'])
        records=[]
        for dt, sub in deficit_news.groupby('changes_baseline_date',sort=True):
            r={'announcement_date':dt,'announcement_quarter':str(dt.to_period('Q')),
               'n_projected_years':len(sub)}
            for shock in ['policy_news_bn','nonpolicy_news_bn','total_revision_bn']:
                r[shock+'_sum_fy']=sub[shock].sum()  # sum FY flow revisions, not a present value
                for g in ['h0_1','h2_5','h6_10','h11plus']:
                    r[shock+'_'+g]=sub.loc[sub.horizon_group.astype(str)==g,shock].sum()
            # abs-weighted centroid to avoid denominator cancellation for signed shocks
            w=sub.policy_news_bn.abs()
            r['policy_news_abs_weighted_mean_horizon']=float((w*sub.horizon_fy).sum()/w.sum()) if w.sum()>0 else np.nan
            r['policy_news_late_share_abs']=float(w.loc[sub.horizon_fy>=2].sum()/w.sum()) if w.sum()>0 else np.nan
            records.append(r)
        timing=pd.DataFrame.from_records(records)
        save('13_fiscal_timing_by_vintage',timing,['announcement_date'])
        sumcols=[c for c in timing if c.endswith('_sum_fy') or c.endswith(('_h0_1','_h2_5','_h6_10','_h11plus'))]
        timing_q=timing.groupby('announcement_quarter',as_index=False)[sumcols].sum()
        counts=timing.groupby('announcement_quarter',as_index=False).agg(n_announcements=('announcement_date','nunique'))
        timing_q=timing_q.merge(counts,on='announcement_quarter',validate='one_to_one')
        save('14_fiscal_timing_quarterly',timing_q,['announcement_quarter'])
        note('13_fiscal_timing_by_vintage','policy_news_bn_sum_fy','Sum of revisions to annual deficits over covered fiscal years','USD bn summed across FY','CBO','Not present value, not a structural identified fiscal shock')
        note('13_fiscal_timing_by_vintage','policy_news_abs_weighted_mean_horizon','Sum(|legislative deficit revision|*h)/Sum(|legislative revision|)','fiscal years','CBO','Magnitude-based news timing, not expected date of fiscal regime transition')

# 07a. CBO historical realized fiscal amounts (fiscal-year x component)
cbo_actuals = read('cbo_actuals')
if cbo_actuals is not None:
    if 'fiscal_year' in cbo_actuals:
        save('18_cbo_actual_fiscal_year',cbo_actuals,
             ['component','category','subcategory','fiscal_year'] if {'component','category','subcategory','fiscal_year'} <= set(cbo_actuals) else None)
        note('18_cbo_actual_fiscal_year','actual_value','Realized CBO fiscal-year amount by component','USD bn','CBO','Fiscal year, not calendar year')

recent=read('cbo_recent_legislative_profiles')
if recent is not None: save('15_recent_legislative_profiles',recent)


# 08. MASTER quarterly: merge only time-series-aligned observables
parts=[]
for p in [macro, flows, debt_q if raw_debt is not None else None]:
    if p is None: continue
    z=p.drop(columns=['period'],errors='ignore').copy()
    existing=set().union(*(set(x.columns) for x in parts)) if parts else set()
    z=z.drop(columns=[c for c in z if c in existing and c!='quarter'])
    parts.append(z)
master=merge_many(parts)
if master is not None:
    if spf_summary is not None and 'survey_quarter' in spf_summary:
        ss=spf_summary.copy().rename(columns={'survey_quarter':'quarter'})
        wanted=['quarter','spf_cpi_inflation_1yr_pct','spf_cpi_inflation_10yr_pct','spf_gdp_deflator_inflation_1yr_pct']
        wanted=[c for c in wanted if c in ss]
        master=master.merge(ss[wanted],on='quarter',how='left',validate='one_to_one')
    if '14_fiscal_timing_quarterly' in PANELS:
        timingq=PANELS['14_fiscal_timing_quarterly'].rename(columns={'announcement_quarter':'quarter'}).copy()
        timingq['quarter']=pd.PeriodIndex(timingq.quarter,freq='Q')
        master=master.merge(timingq,on='quarter',how='left',validate='one_to_one')
        # Keep zeros ONLY for verified absence of CBO announcements in the explicitly covered window.
        for col in timingq.columns:
            if col!='quarter': master=master.rename(columns={col:'cbo_'+col})
    master['period']=master.quarter.astype(str)
    save('16_master_quarterly',master,['quarter'])
    pre=master.loc[master.quarter.between(pd.Period('1970Q1'),pd.Period('2008Q3'))].copy()
    save('17_tanaka_window_1970Q1_2008Q3',pre,['quarter'])

# 09. Export only four research-ready CSV files.
EXPORT = {
    '16_master_quarterly': '01_fiscal_macro_quarterly.csv',
    '03_debt_monthly': '02_government_debt_monthly.csv',
    '08_cbo_vintage_forecasts': '03_cbo_expectations.csv',
    '11_cbo_news_vintage_fy': '04_cbo_news.csv',
}
for key, filename in EXPORT.items():
    df = PANELS.get(key)
    if df is None or df.empty:
        print('Not available:', filename)
        continue
    df = df.copy()
    for col in df.columns:
        if isinstance(df[col].dtype, pd.PeriodDtype):
            df[col] = df[col].astype(str)
    df.to_csv(OUT / filename, index=False, encoding='utf-8-sig')
    print(f'{filename}: {len(df):,} rows × {len(df.columns)} columns')
print('Output folder:', OUT)
